In [37]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 1. Load Combined Dataset

In [38]:
df = pd.read_csv('../data/processed/combined_student_data.csv')

print(f"Dataset loaded: {len(df)} records")
print(f"Original columns: {df.shape[1]}")
print(f"\nFirst few rows:")
df.head()

Dataset loaded: 1044 records
Original columns: 35

First few rows:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,goout,Dalc,Walc,health,absences,G1,G2,G3,subject,risk_category
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,1,1,3,6,5,6,6,math,High
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,1,1,3,4,5,5,6,math,High
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,2,2,3,3,10,7,8,10,math,Medium
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,1,1,5,2,15,14,15,math,Low
4,GP,F,16,U,GT3,T,3,3,other,other,...,2,1,2,5,4,6,10,10,math,Medium


In [39]:
print("All columns in dataset:")
print(df.columns.tolist())

All columns in dataset:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'subject', 'risk_category']


In [40]:
if 'risk_category' in df.columns:
    print("✓ risk_category column found")
    print(df['risk_category'].value_counts())
else:
    print("⚠️ Creating risk_category from G3...")
    df['risk_category'] = df['G3'].apply(
        lambda x: 'High Risk' if x < 10 else ('Medium Risk' if x < 14 else 'Low Risk')
    )
    print("✓ risk_category created")

✓ risk_category column found
risk_category
Medium    520
Low       294
High      230
Name: count, dtype: int64


## 2. Identify Feature 

Before encoding, categorize our features:
- **Numeric features:** Already numbers (age, Medu, Fedu, etc.)
- **Binary categorical:** Two values (school, sex, address, etc.)
- **Multi-category:** More than two values (Mjob, Fjob, reason, guardian)

In [41]:
print("="*60)
print("FEATURE CATEGORIZATION")
print("="*60)

target_cols = ['G1', 'G2', 'G3', 'risk_category']
print(f"\nTarget columns (to exclude): {target_cols}")

numeric_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 
                    'failures', 'famrel', 'freetime', 'goout', 
                    'Dalc', 'Walc', 'health', 'absences']
print(f"\nNumeric features ({len(numeric_features)}): {numeric_features}")

binary_features = ['school', 'sex', 'address', 'famsize', 'Pstatus',
                   'schoolsup', 'famsup', 'paid', 'activities', 
                   'nursery', 'higher', 'internet', 'romantic']
print(f"\nBinary categorical ({len(binary_features)}): {binary_features}")

multi_cat_features = ['Mjob', 'Fjob', 'reason', 'guardian']
print(f"\nMulti-category ({len(multi_cat_features)}): {multi_cat_features}")

subject_feature = ['subject']
print(f"\nSubject feature: {subject_feature}")

FEATURE CATEGORIZATION

Target columns (to exclude): ['G1', 'G2', 'G3', 'risk_category']

Numeric features (13): ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

Binary categorical (13): ['school', 'sex', 'address', 'famsize', 'Pstatus', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Multi-category (4): ['Mjob', 'Fjob', 'reason', 'guardian']

Subject feature: ['subject']


In [42]:
print("\n" + "="*60)
print("MULTI-CATEGORY FEATURE VALUES")
print("="*60)

for feature in multi_cat_features:
    unique_vals = df[feature].unique()
    print(f"\n{feature}: {list(unique_vals)} ({len(unique_vals)} categories)")



MULTI-CATEGORY FEATURE VALUES

Mjob: ['at_home', 'health', 'other', 'services', 'teacher'] (5 categories)

Fjob: ['teacher', 'other', 'services', 'health', 'at_home'] (5 categories)

reason: ['course', 'other', 'home', 'reputation'] (4 categories)

guardian: ['mother', 'father', 'other'] (3 categories)


## 3. Separate Features and Target

CRITICAL: We must exclude G1, G2, and G3 because they are NOT available at the start of the academic year. Our goal is early intervention prediction.

In [43]:
feature_cols = [col for col in df.columns if col not in target_cols]

print(f"Total columns in dataset: {df.shape[1]}")
print(f"Target columns (excluded): {len(target_cols)}")
print(f"Feature columns (to use): {len(feature_cols)}")

X = df[feature_cols].copy()
y = df['risk_category'].copy()

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())

Total columns in dataset: 35
Target columns (excluded): 4
Feature columns (to use): 31

X shape: (1044, 31)
y shape: (1044,)

Target distribution:
risk_category
Medium    520
Low       294
High      230
Name: count, dtype: int64


## 4. One-Hot Encoding

We'll use `pd.get_dummies()` with `drop_first=True` to:
- Convert categorical variables to numeric
- Avoid multicollinearity (dummy variable trap)
- Create ~50-60 total features

In [44]:
print("Data types before encoding:")
print(X.dtypes.value_counts())
print(f"\nCategorical columns: {X.select_dtypes(include='object').columns.tolist()}")

Data types before encoding:
str      18
int64    13
Name: count, dtype: int64

Categorical columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'subject']


In [45]:
# Apply one-hot encoding
print("="*60)
print("ONE-HOT ENCODING")
print("="*60)

# Original shape
print(f"\nOriginal X shape: {X.shape}")

# Apply pd.get_dummies with drop_first=True
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# New shape
print(f"Encoded X shape: {X_encoded.shape}")
print(f"\nFeature explosion: {X.shape[1]} → {X_encoded.shape[1]} columns")
print(f"Added {X_encoded.shape[1] - X.shape[1]} new columns from encoding")

ONE-HOT ENCODING

Original X shape: (1044, 31)
Encoded X shape: (1044, 40)

Feature explosion: 31 → 40 columns
Added 9 new columns from encoding


In [46]:
original_cols = set(X.columns)
new_cols = set(X_encoded.columns)
added_cols = sorted(new_cols - original_cols)

print("\n" + "="*60)
print("NEW COLUMNS CREATED BY ENCODING")
print("="*60)

for col in added_cols:
    print(f"  {col}")

print(f"\nTotal new columns: {len(added_cols)}")


NEW COLUMNS CREATED BY ENCODING
  Fjob_health
  Fjob_other
  Fjob_services
  Fjob_teacher
  Mjob_health
  Mjob_other
  Mjob_services
  Mjob_teacher
  Pstatus_T
  activities_yes
  address_U
  famsize_LE3
  famsup_yes
  guardian_mother
  guardian_other
  higher_yes
  internet_yes
  nursery_yes
  paid_yes
  reason_home
  reason_other
  reason_reputation
  romantic_yes
  school_MS
  schoolsup_yes
  sex_M
  subject_portuguese

Total new columns: 27


In [47]:
# Verify encoding worked correctly
print("="*60)
print("ENCODING VERIFICATION")
print("="*60)

# Check for object types (should be none)
object_cols = X_encoded.select_dtypes(include='object').columns
if len(object_cols) == 0:
    print("✓ No object columns remaining - all categorical variables encoded")
else:
    print(f"⚠️ Warning: Still have object columns: {list(object_cols)}")

# Check data types
print(f"\nData types after encoding:")
print(X_encoded.dtypes.value_counts())

# Check for missing values
print(f"\nMissing values: {X_encoded.isnull().sum().sum()}")

ENCODING VERIFICATION
✓ No object columns remaining - all categorical variables encoded

Data types after encoding:
int64    40
Name: count, dtype: int64

Missing values: 0


In [48]:
# Show sample of encoded data
print("Sample of encoded data:")
X_encoded.head()

Sample of encoded data:


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes,subject_portuguese
0,18,4,4,2,2,0,4,3,4,1,...,0,1,0,0,0,1,1,0,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,1,0,0,0,1,1,0,0
2,15,1,1,1,2,3,4,3,2,2,...,0,1,0,1,0,1,1,1,0,0
3,15,4,2,1,3,0,3,2,2,1,...,0,0,1,1,1,1,1,1,1,0
4,16,3,3,1,2,0,4,3,2,1,...,0,0,1,1,0,1,1,0,0,0


## 5. Create Derived Features

Creating new features that combine existing ones can improve model performance:
- **parent_edu_avg:** Average of mother's and father's education
- **total_alcohol:** Sum of weekday and weekend alcohol consumption
- **has_support:** Whether student has ANY support (school, family, or paid)

In [49]:
# Create derived features
print("="*60)
print("CREATING DERIVED FEATURES")
print("="*60)

# Feature 1: Average parent education
X_encoded['parent_edu_avg'] = (X_encoded['Medu'] + X_encoded['Fedu']) / 2
print("✓ Created parent_edu_avg: (Medu + Fedu) / 2")
print(f"  Range: {X_encoded['parent_edu_avg'].min():.1f} to {X_encoded['parent_edu_avg'].max():.1f}")
print(f"  Mean: {X_encoded['parent_edu_avg'].mean():.2f}")

# Feature 2: Total alcohol consumption
X_encoded['total_alcohol'] = X_encoded['Dalc'] + X_encoded['Walc']
print("\n✓ Created total_alcohol: Dalc + Walc")
print(f"  Range: {X_encoded['total_alcohol'].min()} to {X_encoded['total_alcohol'].max()}")
print(f"  Mean: {X_encoded['total_alcohol'].mean():.2f}")

# Feature 3: Has any support (school OR family OR paid)
# Note: After one-hot encoding with drop_first=True, "yes" becomes 1, "no" becomes 0
# We need to check if the columns exist as they may have been encoded differently

# Check which support columns exist
support_cols = [col for col in X_encoded.columns if 'schoolsup' in col or 'famsup' in col or 'paid' in col]
print(f"\n✓ Support columns found: {support_cols}")

# Create has_support (1 if any support, 0 if none)
# This will work if the categorical variables were encoded
if 'schoolsup' in X_encoded.columns:
    # Direct binary columns
    X_encoded['has_support'] = ((X_encoded['schoolsup'] == 1) | 
                                (X_encoded['famsup'] == 1) | 
                                (X_encoded['paid'] == 1)).astype(int)
elif any('schoolsup' in col for col in X_encoded.columns):
    # Encoded columns (check for _yes suffix or similar)
    schoolsup_col = [col for col in X_encoded.columns if 'schoolsup' in col][0] if any('schoolsup' in col for col in X_encoded.columns) else None
    famsup_col = [col for col in X_encoded.columns if 'famsup' in col][0] if any('famsup' in col for col in X_encoded.columns) else None
    paid_col = [col for col in X_encoded.columns if 'paid' in col][0] if any('paid' in col for col in X_encoded.columns) else None
    
    # Combine them
    has_support = pd.Series(0, index=X_encoded.index)
    if schoolsup_col: has_support |= X_encoded[schoolsup_col]
    if famsup_col: has_support |= X_encoded[famsup_col]
    if paid_col: has_support |= X_encoded[paid_col]
    X_encoded['has_support'] = has_support.astype(int)

print("✓ Created has_support: 1 if any support (school/family/paid), 0 otherwise")
print(f"  Students with support: {X_encoded['has_support'].sum()} ({X_encoded['has_support'].sum()/len(X_encoded)*100:.1f}%)")
print(f"  Students without support: {(X_encoded['has_support']==0).sum()} ({(X_encoded['has_support']==0).sum()/len(X_encoded)*100:.1f}%)")

CREATING DERIVED FEATURES
✓ Created parent_edu_avg: (Medu + Fedu) / 2
  Range: 0.0 to 4.0
  Mean: 2.50

✓ Created total_alcohol: Dalc + Walc
  Range: 2 to 10
  Mean: 3.78

✓ Support columns found: ['schoolsup_yes', 'famsup_yes', 'paid_yes']
✓ Created has_support: 1 if any support (school/family/paid), 0 otherwise
  Students with support: 717 (68.7%)
  Students without support: 327 (31.3%)


In [50]:
# Final feature count
print("\n" + "="*60)
print("FINAL FEATURE SET")
print("="*60)

print(f"\nTotal features after engineering: {X_encoded.shape[1]}")
print(f"Total records: {X_encoded.shape[0]}")

# Breakdown
num_original = len(numeric_features)
num_encoded = X_encoded.shape[1] - 3  # Subtract the 3 derived features
num_derived = 3

print(f"\nBreakdown:")
print(f"  Original numeric features: {num_original}")
print(f"  Features from encoding: {num_encoded}")
print(f"  Derived features: {num_derived}")
print(f"  Total: {X_encoded.shape[1]}")


FINAL FEATURE SET

Total features after engineering: 43
Total records: 1044

Breakdown:
  Original numeric features: 13
  Features from encoding: 40
  Derived features: 3
  Total: 43


## 6. Verify No Target Leakage

CRITICAL: Verify that G1, G2, and G3 are NOT in our feature set (this would be data leakage)

In [51]:
# Check for data leakage
print("="*60)
print("DATA LEAKAGE CHECK")
print("="*60)

leakage_cols = ['G1', 'G2', 'G3']
found_leakage = []

for col in leakage_cols:
    if col in X_encoded.columns:
        found_leakage.append(col)

if len(found_leakage) == 0:
    print("✓ NO DATA LEAKAGE DETECTED")
    print("✓ G1, G2, G3 are NOT in feature set (correct!)")
else:
    print(f"⚠️ WARNING: Data leakage found! Columns: {found_leakage}")
    print("⚠️ These columns must be removed!")

DATA LEAKAGE CHECK
✓ NO DATA LEAKAGE DETECTED
✓ G1, G2, G3 are NOT in feature set (correct!)


## 7. Save Engineered Features

Save the processed features for Week 5 modeling

In [52]:
# Create processed directory if it doesn't exist
import os
os.makedirs('../data/processed', exist_ok=True)

print("="*60)
print("SAVING ENGINEERED FEATURES")
print("="*60)

# Save features
X_encoded.to_csv('../data/processed/engineered_features.csv', index=False)
print(f"✓ Saved features to: data/processed/engineered_features.csv")
print(f"  Shape: {X_encoded.shape}")

# Save target
y.to_csv('../data/processed/target.csv', index=False)
print(f"✓ Saved target to: data/processed/target.csv")
print(f"  Shape: {y.shape}")

# Save feature names (important for prediction later!)
feature_names = X_encoded.columns.tolist()
with open('../data/processed/feature_names.txt', 'w') as f:
    for name in feature_names:
        f.write(f"{name}\n")
print(f"✓ Saved feature names to: data/processed/feature_names.txt")
print(f"  Total features: {len(feature_names)}")

SAVING ENGINEERED FEATURES
✓ Saved features to: data/processed/engineered_features.csv
  Shape: (1044, 43)
✓ Saved target to: data/processed/target.csv
  Shape: (1044,)
✓ Saved feature names to: data/processed/feature_names.txt
  Total features: 43


In [53]:
# Verify files were saved
import os

files_to_check = [
    '../data/processed/engineered_features.csv',
    '../data/processed/target.csv',
    '../data/processed/feature_names.txt'
]

print("\n" + "="*60)
print("FILE VERIFICATION")
print("="*60)

for file in files_to_check:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # Size in KB
        print(f"✓ {file} ({size:.1f} KB)")
    else:
        print(f"✗ {file} - NOT FOUND!")


FILE VERIFICATION
✓ ../data/processed/engineered_features.csv (91.4 KB)
✓ ../data/processed/target.csv (5.8 KB)
✓ ../data/processed/feature_names.txt (0.5 KB)


## 8. Summary

In [54]:
print("="*60)
print("FEATURE ENGINEERING SUMMARY")
print("="*60)

print(f"\n✅ COMPLETED:")
print(f"  • Loaded 1,044 student records")
print(f"  • One-hot encoded categorical variables")
print(f"  • Created 3 derived features")
print(f"  • Excluded G1, G2, G3 (no data leakage)")
print(f"  • Saved engineered features")

print(f"\n📊 RESULTS:")
print(f"  • Original features: {X.shape[1]}")
print(f"  • Engineered features: {X_encoded.shape[1]}")
print(f"  • Feature explosion: {X.shape[1]} → {X_encoded.shape[1]}")

print(f"\n💾 FILES SAVED:")
print(f"  • data/processed/engineered_features.csv ({X_encoded.shape[0]} rows × {X_encoded.shape[1]} cols)")
print(f"  • data/processed/target.csv ({y.shape[0]} rows)")
print(f"  • data/processed/feature_names.txt ({len(feature_names)} features)")


print(f"\n✓ Feature engineering complete!")
print("="*60)

FEATURE ENGINEERING SUMMARY

✅ COMPLETED:
  • Loaded 1,044 student records
  • One-hot encoded categorical variables
  • Created 3 derived features
  • Excluded G1, G2, G3 (no data leakage)
  • Saved engineered features

📊 RESULTS:
  • Original features: 31
  • Engineered features: 43
  • Feature explosion: 31 → 43

💾 FILES SAVED:
  • data/processed/engineered_features.csv (1044 rows × 43 cols)
  • data/processed/target.csv (1044 rows)
  • data/processed/feature_names.txt (43 features)

✓ Feature engineering complete!
